### Text preprocess

In [ ]:
import nltk
from nltk import tokenize
import numpy as np 
from string import punctuation
import unidecode
import pandas as pd
#stemmer = nltk.RSLPStemmer()


def proccess_text(tweets):
    
    # Removing links, mentions and hashtags
    tweets['processed_text'] = tweets.text.str.replace(r'(http\S+)', '',regex=True) \
                                          .str.replace(r'@[\w]*', '',regex=True) \
                                          .str.replace(r'#[\w]*','',regex=True) 
    print('[ok] - Removing links.')
    print('[ok] - Removing mentions.')
    print('[ok] - Removing hashtags.')

    textWords = ' '.join([text for text in tweets.processed_text])

    # Removing accent
    textWords = [unidecode.unidecode(text) for text in tweets.processed_text ]    
    print('[ok] - Removing accent.')
    
    # Creating a list of words and characters (stopwords) to be removed from the text
    # stopWords = nltk.corpus.stopwords.words("portuguese")    
    print('[ok] - Creating a list of words and characters (stopwords) to be removed from the text.')
    
    
    # Separating punctuation from words
    punctSeparator = tokenize.WordPunctTokenizer()
    punctuationList = list()
    for punct in punctuation:
        punctuationList.append(punct)
        
    #stopWords =   punctuationList + stopWords    
    stopWords =   punctuationList
    #print('[ok] - Separating punctuation from words.')


    # Iterating over the text and removing stop words 
    trasnformedText = list()    
    for text in textWords:
        newText = list()   
        text = text.lower()
        textWords = punctSeparator.tokenize(text)
        for words in textWords:
             if words not in stopWords:
                #newText.append(stemmer.stem(words))
                newText.append(words)
        trasnformedText.append(' '.join(newText))
    tweets.processed_text = trasnformedText
    print('[ok] - Removing punctuation and set text to lowecase.')
   
    # Removing all non-text characters
    tweets.processed_text = tweets['processed_text'].str.replace(r"[^a-zA-Z#]", " ", regex=True)                                                         
    print('[ok] - Removing all non-text characters.')
   
    trasnformedText = list()
    for phrase in tweets.processed_text:
        newPhrase = list()   
        newPhrase.append(' '.join(phrase.split()))
        for words in newPhrase:
            trasnformedText.append(''.join(newPhrase))
    tweets.processed_text = trasnformedText
    
    # Removing tweets with less than three terms
    index=[x for x in tweets.index if tweets.processed_text[x].count(' ') < 3]
    tweets = tweets.drop(index)
    print('[ok] - Removing tweets with less than three terms.')

    # Removing empty lines
    removeEmpty  = tweets.processed_text != ' '
    tweets = tweets[removeEmpty]
    print('[ok] - Removing empty lines.')

    tweets.reset_index(inplace=True)
    #tweets = {'created_at': tweets.created_at, 'id':tweets.id,'author_id':tweets.author_id,'in_reply_to_user_id':tweets.in_reply_to_user_id, 'text': tweets.processed_text}
    tweets = {'text': tweets.processed_text}
    tweets = pd.DataFrame(tweets)
    #tweets = tweets.sort_values(['created_at']).reset_index().drop(columns=["index"])
    #tweets = tweets.reset_index().drop(columns=["index"])
    
    return tweets

In [ ]:
import pandas as pd 

trump2016 = pd.read_csv('../data/raw/trump2016.csv')
trump2016 = trump2016[['id','text']]
trump2016 = trump2016.dropna()



donaldtrump = pd.read_csv(open('../data/raw/donaldtrump.csv','rU'), encoding='utf-8')
donaldtrump = donaldtrump[['id','text']]
donaldtrump = donaldtrump.dropna()


trump = pd.concat([trump2016,donaldtrump])
trump = trump.reset_index().drop(columns=['index'])
len(trump)

In [ ]:
marcelo = pd.read_csv('../data/raw/trump_raw_marcelo.csv')

trump = trump[~trump.id.isin(marcelo.id)]
len(trump)

In [ ]:

trump_marcelo_concact = pd.concat([trump,marcelo])
trump_marcelo_concact = trump_marcelo_concact.reset_index().drop(columns=['index'])
len(trump_marcelo_concact)
trump_marcelo_concact.to_csv('../data/raw/trump_marcelo_concact.csv', index=False)

In [ ]:
tweets = proccess_text(trump_marcelo_concact)

In [ ]:
tweets.to_csv('../data/processed/trump_marcelo_processed.csv', index=False)

### Extract more frequent hashtags

In [1]:
import re
import nltk
from nltk import tokenize

def count_hashtags(data_of_text):
    regex = re.compile(r'#[\w]*')
    #regex = re.compile(r'#(\w*[0-9a-zA-Z]+\w*[0-9a-zA-Z])')

    textWords = ' '.join([text for text in data_of_text])

    hashtags = regex.findall(textWords)

    hashtags = ' '.join([text for text in  hashtags])

    tokenizing = tokenize.WhitespaceTokenizer()
    tokenizedWords = tokenizing.tokenize(hashtags)
    frequency = nltk.FreqDist(tokenizedWords)
    df_frequency = pd.DataFrame({"Hashtag": list(frequency.keys()),
                                       "Frequency": list(frequency.values())})

    return df_frequency

In [2]:
import nltk
from nltk import tokenize
import numpy as np 
from string import punctuation
import unidecode


    #tokens = tokenize.WordPunctTokenizer()
    #tokens = tokenize.WhitespaceTokenizer()
    #tokens = tokenize.MWETokenizer()

def proccess_text(data_of_text, tk):
    textWords = [unidecode.unidecode(text) for text in data_of_text]       
    
    
    punctuationList = list()
    for punct in punctuation:
        if punct != '#':
            punctuationList.append(punct)
    trasnformedText = list()
    
    personalList=["...","!#","@#","'#",".#","\"#","...#","(#","?#","!!"]  
    punctuationList = punctuationList  
    
    for text in textWords:
        newText = list()   
        text = text.lower()
        textWords = tk.tokenize(text)
        for words in textWords: 
             if words not in punctuationList:
                 newText.append(words)
        trasnformedText.append(' '.join(newText))
    return trasnformedText

In [3]:
import pandas as pd

tweets = pd.read_csv('../data/raw/trump_marcelo_concact.csv')
tweets

,id,text
0,8.149840e+17,What does Trump mean to You?💖 https://t.co/L2...
1,8.149833e+17,#AmericaIsAlreadyGreat https://t.co/2xzu8eWSeK...
2,8.149809e+17,@infowars #Trump2016 #MakeAmericaGreatAgain Is...
3,8.149794e+17,"""The Thin Blue Line is under attack"" by Chris ..."
4,8.149789e+17,#TuckFrump: The Video Making People #VoteTrump...
...,...,...
457083,6.216854e+17,\tFighting 2 make America Great Again I look f...
457084,6.216855e+17,\tWhat if all the controversy over the #Donald...
457085,6.216856e+17,\tI just heard on the radio that Trump is #1 i...
457086,6.216857e+17,\tThe @realDonaldTrump doesn't do the PC tap d...


In [4]:
tweets['processed_text'] = proccess_text(tweets.text, tokenize.WhitespaceTokenizer())
tweets


,id,text,processed_text
0,8.149840e+17,What does Trump mean to You?💖 https://t.co/L2...,what does trump mean to you? https://t.co/l2ch...
1,8.149833e+17,#AmericaIsAlreadyGreat https://t.co/2xzu8eWSeK...,#americaisalreadygreat https://t.co/2xzu8ewsek...
2,8.149809e+17,@infowars #Trump2016 #MakeAmericaGreatAgain Is...,@infowars #trump2016 #makeamericagreatagain is...
3,8.149794e+17,"""The Thin Blue Line is under attack"" by Chris ...","""the thin blue line is under attack"" by chris ..."
4,8.149789e+17,#TuckFrump: The Video Making People #VoteTrump...,#tuckfrump: the video making people #votetrump...
...,...,...,...
457083,6.216854e+17,\tFighting 2 make America Great Again I look f...,fighting 2 make america great again i look for...
457084,6.216855e+17,\tWhat if all the controversy over the #Donald...,what if all the controversy over the #donaldtr...
457085,6.216856e+17,\tI just heard on the radio that Trump is #1 i...,i just heard on the radio that trump is #1 in ...
457086,6.216857e+17,\tThe @realDonaldTrump doesn't do the PC tap d...,the @realdonaldtrump doesn't do the pc tap dan...


In [5]:
df = count_hashtags(tweets.processed_text)
df

,Hashtag,Frequency
0,#trump2016,256563
1,#veterans,525
2,#,3229
3,#americaisalreadygreat,2101
4,#makedonalddrumpfagain,2275
...,...,...
75068,#obamasux,1
75069,#donaldumb,1
75070,#trumpbuildthatwall,1
75071,#kidsreadyfortrump,1


In [7]:
df.head()

,Hashtag,Frequency
0,#trump2016,256563
1,#veterans,525
2,#,3229
3,#americaisalreadygreat,2101
4,#makedonalddrumpfagain,2275


In [6]:
import tabloo

tabloo.show(df)

'FLASK_ENV' is deprecated and will not be used in Flask 2.3. Use 'FLASK_DEBUG' instead.
'FLASK_ENV' is deprecated and will not be used in Flask 2.3. Use 'FLASK_DEBUG' instead.


KeyError: 'WERKZEUG_SERVER_FD'